## Deliverable 2
Markus Valland, Jørgen Skarra, Wiktor Correia-Myrhagen



### Task 1: File ingestion and regular expressions

In [1]:

import re
import json
from pathlib import Path

import numpy as np
import pandas as pd


#Task 1 File ingestion and regular expressions
def read_file(path) -> tuple[str, str]:
    raw = Path(path).read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("iso-8859-1"), "iso-8859-1"

#Compile regex pattern for log line parsing 
LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
)

#Function to parse log lines into structured records
def parse_log(text: str) -> list[dict]:
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip().lstrip("\ufeff"))
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

### Task 2: Recursive utility functions

In [2]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    #Finds all the logfiles
    directory = Path(directory) # converts
    log_files = [] 
    for item in directory.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item, extension))  # recursive call, looks into the subfolders
        elif item.is_file() and item.suffix == extension: #only includes files ending .log
            log_files.append(item)
    return log_files

In [3]:
log_files = find_log_files("data")
print(f"Found {len(log_files)} log files")
for f in log_files:
    print(f) #print log files

Found 9 log files
data/2026/science_building/POS_01/register_sales.log
data/2026/science_building/POS_03/register_sales.log
data/2026/science_building/POS_02/register_sales.log
data/2026/student_union/POS_01/register_sales.log
data/2026/student_union/POS_03/register_sales.log
data/2026/student_union/POS_02/register_sales.log
data/2026/library_cafe/POS_01/register_sales.log
data/2026/library_cafe/POS_03/register_sales.log
data/2026/library_cafe/POS_02/register_sales.log


In [4]:
def find_active_registers(data) -> set[str]:
    """Looks through JSON file and collect ID of active registers."""
    active = set()
    if isinstance(data, dict): #looks into dictonaries
        if data.get("active") is True and "id" in data:
            active.add(data["id"])
        for value in data.values():
            active |= find_active_registers(value)   # recursive call
    elif isinstance(data, list): # looks into lists
        for element in data:
            active |= find_active_registers(element)  # recursive call
    return active

In [5]:
config = json.loads(Path("data/store_config.json").read_text(encoding="utf-8"))
active_registers = find_active_registers(config)
print(sorted(active_registers))
#finds active registers in store_config.json and writes them out sorted

['POS_01', 'POS_02', 'POS_03']


## Task 3: Cleaning and analysing data using NumPy


In [6]:
def remove_price_outliers(prices) -> np.ndarray: #remove outliers
    """Replaces values more than 3 standard deviations from the mean
    with the median of the valid prices. Returns a cleaned 1D array."""

    prices = np.array(prices, dtype=float).ravel()

    mean = prices.mean()
    std = prices.std()

    # Outliers deviate from the mean by more than 3 standard deviations
    is_outlier = np.abs(prices - mean) > 3 * std

    # The median is calculated from the valid prices only
    median = np.median(prices[~is_outlier])

    prices[is_outlier] = median
    return prices

In [7]:
#test file
test_prices = [4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 999.0]
cleaned = remove_price_outliers(test_prices)
print("Before:", test_prices)
print("After: ", cleaned)

Before: [4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 999.0]
After:  [4.5 3.  4.2 3.5 3.8 7.5 4.5 3.  4.2 3.5 3.8 7.5 4. ]


In [8]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Computes the element-wise refund ratio using NumPy array operations."""
    epsilon = 1e-6  # avoids division by zero when sales_count is 0
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + epsilon)

In [9]:
sales = np.array([10, 5, 0])
refunds = np.array([2, 1, 0])
print(compute_refund_ratio(sales, refunds))

[0.19999998 0.19999996 0.        ]


## Task 4: Data analysis using Pandas

In [10]:
STATUSES = ["SUCCESS", "REFUND", "CANCELLED"]


def transactions_to_dataframe(records: list[dict]) -> pd.DataFrame:
    """Converts the list of transaction dictionaries to a DataFrame
    with a sorted datetime index."""
    df = pd.DataFrame(records)                          # list of dicts -> table
    df["timestamp"] = pd.to_datetime(df["timestamp"],   # text -> datetime
                                     format="%Y-%m-%d %H:%M:%S")
    df = df.set_index("timestamp")                      # timestamp as index
    return df.sort_index()                              # sort by time


def hourly_revenue(df: pd.DataFrame) -> pd.Series:
    """Groups by item and sums the price in 1-hour windows."""
    return df.groupby("item")["price"].resample("1h").sum()  # sum price per item per hour


def status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Total revenue per item (rows) for each status (columns)."""
    pivot = df.pivot_table(index="item",       # rows
                           columns="status",   # columns
                           values="price",     # what to sum
                           aggfunc="sum",
                           fill_value=0.0)
    # makes sure all three status columns exist and are in a fixed order
    return pivot.reindex(columns=STATUSES, fill_value=0.0)


def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Counts sales (SUCCESS) and refunds (REFUND) per item and
    calculates the refund ratio with compute_refund_ratio from task 3."""
    counts = pd.crosstab(df["item"], df["status"]).reindex(columns=STATUSES, fill_value=0)  # count rows per item/status
    sales_count = counts["SUCCESS"].to_numpy()
    refund_count = counts["REFUND"].to_numpy()
    return pd.DataFrame({
        "sales_count": sales_count,
        "refund_count": refund_count,
        "refund_ratio": compute_refund_ratio(sales_count, refund_count),  # from task 3
    }, index=counts.index)

In [11]:
#test of task 4 on the raw (not yet cleaned) data
records = []
for f in log_files:
    text, encoding = read_file(f)
    records.extend(parse_log(text))

df = transactions_to_dataframe(records)
print(df.head(), "\n")
print(hourly_revenue(df).head(10), "\n")
print(status_pivot_table(df), "\n")
print(item_refund_ratios(df))

                    register_id        item  price currency   status
timestamp                                                           
2026-10-15 07:00:17      POS_03  Cappuccino   4.65      EUR  SUCCESS
2026-10-15 07:00:19      POS_01       Latte   4.36      USD  SUCCESS
2026-10-15 07:00:30      POS_03    Espresso   3.37      USD  SUCCESS
2026-10-15 07:00:58      POS_03  Cappuccino   4.20      EUR   REFUND
2026-10-15 07:01:06      POS_01    Sandwich   7.78      USD  SUCCESS 

item        timestamp          
Cappuccino  2026-10-15 07:00:00    2236.02
            2026-10-15 08:00:00     207.81
            2026-10-15 09:00:00    1031.52
Croissant   2026-10-15 07:00:00    4217.14
            2026-10-15 08:00:00    1185.02
            2026-10-15 09:00:00      42.03
Espresso    2026-10-15 07:00:00     161.50
            2026-10-15 08:00:00    4142.40
            2026-10-15 09:00:00      41.31
Latte       2026-10-15 07:00:00     261.75
Name: price, dtype: float64 

status      SUCCESS   R

## Task 5: Full workflow

In [12]:
#Step 1: find all log files and the active registers
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)  # creates the folder if it does not exist

log_files = find_log_files(DATA_DIR)  # task 2: recursive search for all .log files
config = json.loads((DATA_DIR / "store_config.json").read_text(encoding="utf-8"))  # JSON text -> nested dicts/lists
active_registers = find_active_registers(config)  # task 2: set of register ids with "active": true

print(f"Found {len(log_files)} log files")
print("Active registers:", sorted(active_registers))

Found 9 log files
Active registers: ['POS_01', 'POS_02', 'POS_03']


In [13]:
#Step 2: read every file and parse the valid lines
records = []
for file in log_files:
    text, encoding = read_file(file)  # task 1: tries utf-8 first, falls back to iso-8859-1
    file_records = parse_log(text)    # task 1: regex keeps sales lines, skips noise lines
    print(f"{file}: {encoding}, {len(file_records)} transactions")
    records.extend(file_records)      # extend adds each dict, append would add the whole list as one element

# keep only transactions from active registers (key name comes from the regex group)
records = [r for r in records if r["register_id"] in active_registers]
print("Total transactions:", len(records))

data/2026/science_building/POS_01/register_sales.log: utf-8, 80 transactions
data/2026/science_building/POS_03/register_sales.log: utf-8, 80 transactions
data/2026/science_building/POS_02/register_sales.log: utf-8, 80 transactions
data/2026/student_union/POS_01/register_sales.log: utf-8, 80 transactions
data/2026/student_union/POS_03/register_sales.log: utf-8, 80 transactions
data/2026/student_union/POS_02/register_sales.log: utf-8, 80 transactions
data/2026/library_cafe/POS_01/register_sales.log: utf-8, 80 transactions
data/2026/library_cafe/POS_03/register_sales.log: utf-8, 80 transactions
data/2026/library_cafe/POS_02/register_sales.log: utf-8, 80 transactions
Total transactions: 720


In [14]:
#Step 3: build the DataFrame
df = transactions_to_dataframe(records)  # task 4: datetime index, sorted chronologically
df.head()

,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:17,POS_03,Cappuccino,4.65,EUR,SUCCESS
2026-10-15 07:00:19,POS_01,Latte,4.36,USD,SUCCESS
2026-10-15 07:00:30,POS_03,Espresso,3.37,USD,SUCCESS
2026-10-15 07:00:58,POS_03,Cappuccino,4.20,EUR,REFUND
2026-10-15 07:01:06,POS_01,Sandwich,7.78,USD,SUCCESS


In [15]:
#Step 4: replace price glitches (e.g. 999.00) with the median
# done before the NOK correction, so all prices are on the same scale
# and the real NOK prices are not mistaken for outliers
print("Max price before:", df["price"].max())
df["price"] = remove_price_outliers(df["price"])  # task 3: |x - mean| > 3 std -> median
print("Max price after: ", df["price"].max())

Max price before: 999.0
Max price after:  7.98


In [16]:
#Step 5: NOK prices are registered on EUR scale, multiply them by 10
# NB: only run this cell once, every run multiplies NOK by 10 again
is_nok = df["currency"] == "NOK"  # boolean mask, True for NOK rows
df.loc[is_nok, "price"] *= 10     # .loc changes df itself (rows = mask, column = price)

print(df.groupby("currency")["price"].mean())  # NOK should now be about 10x the others

currency
EUR     4.307309
NOK    45.140444
USD     4.309715
Name: price, dtype: float64


In [17]:
#Step 6: generate the reports with the task 4 functions
hourly = hourly_revenue(df)             # Series: (item, hour) -> total price
pivot = status_pivot_table(df)          # DataFrame: item x status -> total price
refund_ratios = item_refund_ratios(df)  # DataFrame: sales, refunds and ratio per item

print(hourly.head(), "\n")
print(pivot, "\n")
print(refund_ratios)

item        timestamp          
Cappuccino  2026-10-15 07:00:00    1054.08
            2026-10-15 08:00:00     846.36
            2026-10-15 09:00:00     110.34
Croissant   2026-10-15 07:00:00     950.02
            2026-10-15 08:00:00     623.93
Name: price, dtype: float64 

status      SUCCESS  REFUND  CANCELLED
item                                  
Cappuccino  1385.21  425.54     200.03
Croissant   1014.27  391.26     314.85
Espresso     726.97  151.86     373.59
Latte       1396.43  384.23     254.38
Muffin      1045.04  270.35     220.66
Sandwich    2602.08  635.93     496.63 

            sales_count  refund_count  refund_ratio
item                                               
Cappuccino           75            21      0.280000
Croissant            79            24      0.303797
Espresso             89            11      0.123596
Latte                93            22      0.236559
Muffin               77            14      0.181818
Sandwich             79            22      0.

In [18]:
#Step 7: save all output files
df.to_csv(OUTPUT_DIR / "cleaned_transactions.csv")           # the timestamp index is written as the first column
pivot.to_csv(OUTPUT_DIR / "status_pivot_table.csv")
refund_ratios.to_csv(OUTPUT_DIR / "item_refund_ratios.csv")

# JSON cannot store a pandas Series directly, so we build a nested dict:
# {"Latte": {"2026-10-15 07:00": 123.45, ...}, ...}
hourly_report = {}
for (item, hour), total in hourly.items():  # the index has two levels: (item, hour)
    # setdefault creates an empty dict the first time an item is seen
    # strftime turns the datetime into text, since JSON cannot store datetime objects
    hourly_report.setdefault(item, {})[hour.strftime("%Y-%m-%d %H:%M")] = round(total, 2)

with open(OUTPUT_DIR / "hourly_revenue.json", "w", encoding="utf-8") as f:
    json.dump(hourly_report, f, indent=4)  # dump writes to a file, dumps would return a string

print("Saved:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

Saved: ['cleaned_transactions.csv', 'hourly_revenue.json', 'item_refund_ratios.csv', 'status_pivot_table.csv']


## Discussion

### Summary of the task
The goal was to build a pipeline for the Campus Coffee Shop chain that turns messy register logs into clean data and reports:

1. **File ingestion (Task 1):** `read_file` tries UTF-8 first and falls back to ISO-8859-1, so the encoding is never hard-coded. `parse_log` uses a regular expression with named groups to pick out the 6 fields from valid sales lines and skip printer/network noise.
2. **Recursion (Task 2):** `find_log_files` searches the nested `year/store/register` folders with explicit recursion (no `glob`/`os.walk`), and `find_active_registers` walks the nested JSON config to find all registers with `"active": true`.
3. **NumPy (Task 3):** `remove_price_outliers` replaces prices more than 3 standard deviations from the mean with the median of the valid prices, and `compute_refund_ratio` computes refunds / (sales + 10⁻⁶).
4. **Pandas (Task 4):** the records are turned into a DataFrame with a sorted datetime index, which is used for hourly revenue per item, a pivot table of revenue per status, and refund ratios per item.
5. **Workflow (Task 5):** all functions are combined into seven steps, and the results are saved to the `output/` folder.

### Results
- **9 log files** were found, and all three registers (POS_01, POS_02, POS_03) are active, giving **720 transactions** in total.
- All files were reported as UTF-8, even though the data generator writes some of them as ISO-8859-1. The reason is that the log lines only contain ASCII characters, which are stored as identical bytes in both encodings. The fallback to ISO-8859-1 is therefore only used when a file contains non-ASCII bytes (e.g. ø or é), which is not the case here.
- Outlier cleaning replaced all **17 hardware glitches** (price 999.00) with the median: the maximum price went from **999.00 to 7.98**.
- After the currency correction the mean NOK price is about **45**, compared to about **4.3** for EUR and USD, which shows that the NOK prices are now on the correct scale.
- The refund ratio is highest for **Croissant (0.30)** and **Cappuccino (0.28)**, and lowest for **Espresso (0.12)**.
- Sandwich has the highest SUCCESS revenue, mostly because it is the most expensive item.

The four output files `cleaned_transactions.csv`, `hourly_revenue.json`, `status_pivot_table.csv` and `item_refund_ratios.csv` are saved in `output/`.

### Discussion and limitations
- **Order of cleaning:** outliers are removed *before* the NOK correction. While all prices are still on the same (EUR) scale, the 999.00 glitches stand out clearly. If NOK prices were multiplied by 10 first, the data would contain two price scales (≈4 and ≈45), which inflates the standard deviation and makes the 3σ rule less reliable. Cleaning first also means that a glitch in a NOK row is replaced by a EUR-scale median, which is then correctly multiplied by 10 in the next step.
- **Mixed currencies:** the revenue in the hourly report and pivot table sums prices in NOK, EUR and USD without converting them to one currency. The totals are therefore not directly comparable, and a natural next step would be to convert everything to one currency using exchange rates.
- **Outlier method:** the mean and standard deviation are themselves affected by the extreme values, and the rule is applied to all items together. A method based on the median (or applied per item) would be more robust.
- **Refund ratio:** we define sales as transactions with status SUCCESS. CANCELLED transactions are not counted as either sales or refunds.
- **Re-running cells:** the NOK correction multiplies the prices in place, so Step 5 must only be run once after Step 3. Wrapping the workflow in a single function would avoid this.

## AI-Summary

Task 1
	Provided the regex pattern (LOG_PATTERN) with named capture groups that extracts the 6 fields and skips the noise lines.
	Checked our read_file function for handling both UTF-8 and ISO-8859-1, and confirmed that BOM characters are removed before parsing.
Task 2 
	Task 2
	Helped write find_log_files and find_active_registers with explicit recursion, and explained how the recursion works. 
Task 3
	Helped implement remove_price_outliers, which replaces values more than 3 standard deviations from the mean with the median of the valid prices.
	Helped write compute_refund_ratio with ϵ=10^(-6) to avoid division by zero.
Task 4
	Suggested the four Pandas functions (DataFrame with datetime index, hourly revenue, status pivot table, refund ratios) and explained them step by step.
	At our request, turned the code into a fill-in-the-blanks exercise, which we completed and tested ourselves.
Task 5
	Set up the workflow as seven steps with blanks that we filled in, and explained the mistakes we made along the way.
	Wrote the export of the hourly revenue to JSON, explained why outliers are removed before the NOK correction, and added comments to the code.
General
	The AI could not run the code. We ran the notebook ourselves and verified the results and the four output files.

## Github Link
